# Fault Tolerance Experiments: Empirical Resilience Verification

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 05_fault_tolerance_experiments.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Laboratory suite of empirical fault tolerance experiments: lineage reconstruction, task failure recovery, and checkpoint restoration.
- **Why does this topic exist?**: Proving resilience empirically prepares you for production chaos engineering.
- **What problem does it solve?**: Assuming fault tolerance works without verifying recovery behavior.
- **Where is it used in real-world Data Engineering?**: Disaster recovery testing for mission-critical pipelines.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("FT_Experiments_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print(f"Loaded {orders.count()} orders for resilience experiments.")

---

## Experiment: Simulated Node Crash Recomputation

- **1. What are we testing?**: Unpersist an RDD and observe that subsequent actions re-evaluate from source files.
- **2. What do we expect?**: Recomputation succeeds seamlessly.

In [ ]:
rdd_test = orders.rdd.map(lambda r: (r[7], 1)).reduceByKey(lambda a, b: a+b).persist()
rdd_test.count() # materialize
rdd_test.unpersist() # simulate block loss
print("Count after simulated loss:", rdd_test.count())

**3. What happened?**: Spark seamlessly re-read source files and re-executed the pipeline.  
**4. Why did it happen?**: Lineage remained intact, allowing automatic recomputation.  
**5. Practical Engineering Lesson**: Spark handles in-memory block loss transparently through lineage graphs.

---

## Practical Exercises

### Exercise 1 (Beginner): Verify Unpersist
**Problem Statement**: Check is_cached is False after unpersist.  
**Input / Context**: rdd_test

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Task Failures in UI
**Problem Statement**: Inspect failed tasks count in UI Executors tab.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Chaos Engineering Scenario
**Problem Statement**: Explain what happens if Master container dies.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Verify Unpersist):
```python
print('Is cached:', rdd_test.is_cached)
```

#### Solution 2 (Check Task Failures in UI):
```python
print('Inspected localhost:4040/executors')
```

#### Solution 3 (Chaos Engineering Scenario):
```python
print('Running applications continue; new submissions block until Master recovers.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Resilience Validated**: Lineage recomputes lost blocks; task retries absorb glitches.

### Essential Technical Terms
- `Chaos Engineering`: Testing systems by intentionally introducing failures.

### Common Engineering Mistakes
- **Assuming data loss on executor crash**: Lineage guarantees deterministic recovery.

### Interview Questions on This Topic
- **Q: Can Spark recover if the Driver dies in client mode?**
  - *A*: No, in client mode the driver hosts the DAGScheduler and SparkContext; if it dies, the application terminates.

### What to Study Next
Module 11 complete! Proceed to Module 12: Advanced Topics.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')